# 05 — Ensemble
Combine per-model submissions via weighted average (weights in `configs/config.yaml`), write `submission_ensemble.csv`.

Requires notebooks 02, 03, 04 to have been run first.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.models import minmax

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
weights = cfg['ensemble']['weights']

In [ ]:
subs = ROOT / 'outputs' / 'submissions'
dfs = {k: pd.read_csv(subs / f'submission_{k}.csv') for k in weights}
ids = dfs[next(iter(weights))]['ID']
mat = np.stack([dfs[k]['TARGET'].to_numpy() for k in weights], axis=0)
w = np.array([weights[k] for k in weights], dtype=np.float64); w /= w.sum()
final = minmax(w @ mat)

In [ ]:
pd.DataFrame({'ID': ids, 'TARGET': final}).to_csv(subs / 'submission_ensemble.csv', index=False)